In [11]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.linear_model import LogisticRegressionCV
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta
import seaborn as sns
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

## Description goes here

In [4]:
# load the training data
train = pd.read_parquet('data/training.parquet')
train.head()

,level_0,index,Month,bond_log_return,bond_price_max,bond_price_min,return_label,bond_max_min_return_diff,DATE,bond,...,Manufacturing_Production_log_return_discretized,Bitcoin_spot_price_log_return_sum,Bitcoin_spot_price_max,Bitcoin_spot_price_min,Bitcoin_spot_price_log_return_discretized,Bitcoin_spot_price_max_min_return_diff,Money_supply,Money_supply_log_return_sum,Money_supply_log_return_discretized,Term
0,0,0,2021-09-01,0.000350,99.918,99.883,Increase,0.000350,2021-09-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Decrease,-0.071135,52691.21,40656.14,Decrease,0.259299,20979.01,0.008040,Increase,2
1,1,1,2021-10-01,-0.003168,99.969,99.574,Decrease,0.003959,2021-10-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,0.335915,66005.18,47692.86,Increase,0.324952,21142.51,0.007763,Increase,2
2,2,2,2021-11-01,0.000040,99.781,99.402,Increase,0.003806,2021-11-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,-0.075391,67510.07,53839.40,Decrease,0.226271,21316.91,0.008215,Increase,2
3,3,3,2021-12-01,-0.003026,99.531,99.246,Decrease,0.002868,2021-12-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,Increase,-0.206989,57228.48,46240.16,Decrease,0.213203,21471.11,0.007208,Increase,2
25,25,0,2021-08-01,0.000782,99.844,99.758,Increase,0.000862,2021-08-01,United States Treasury Notes 0.125% 31-AUG-202...,...,Decrease,0.126516,49464.67,38207.28,Increase,0.258233,20811.01,0.008813,Increase,2


In [6]:
# convert the increase column to binary
# discritized_vars = [x for x in train.columns if "_discretized" in x]
# for col in discritized_vars:
#     train[col] = train[col].apply(lambda x: 1 if x == "Increase" else 0)

train["return_label"] = train["return_label"].apply(lambda x: 1 if x == "Increase" else 0)
train.head()


,level_0,index,Month,bond_log_return,bond_price_max,bond_price_min,return_label,bond_max_min_return_diff,DATE,bond,...,Manufacturing_Production_log_return_discretized,Bitcoin_spot_price_log_return_sum,Bitcoin_spot_price_max,Bitcoin_spot_price_min,Bitcoin_spot_price_log_return_discretized,Bitcoin_spot_price_max_min_return_diff,Money_supply,Money_supply_log_return_sum,Money_supply_log_return_discretized,Term
0,0,0,2021-09-01,0.000350,99.918,99.883,1,0.000350,2021-09-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,0,-0.071135,52691.21,40656.14,0,0.259299,20979.01,0.008040,1,2
1,1,1,2021-10-01,-0.003168,99.969,99.574,0,0.003959,2021-10-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,0.335915,66005.18,47692.86,1,0.324952,21142.51,0.007763,1,2
2,2,2,2021-11-01,0.000040,99.781,99.402,1,0.003806,2021-11-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,-0.075391,67510.07,53839.40,0,0.226271,21316.91,0.008215,1,2
3,3,3,2021-12-01,-0.003026,99.531,99.246,0,0.002868,2021-12-01,United States Treasury Notes 0.25% 30-SEP-2023...,...,1,-0.206989,57228.48,46240.16,0,0.213203,21471.11,0.007208,1,2
25,25,0,2021-08-01,0.000782,99.844,99.758,1,0.000862,2021-08-01,United States Treasury Notes 0.125% 31-AUG-202...,...,0,0.126516,49464.67,38207.28,1,0.258233,20811.01,0.008813,1,2


In [ ]:
# create a function for the feature selection process
def en_feature_selection(x: list[str], y: str, data: pd.DataFrame) -> list[str]:
    """
    Use Elastic Net to select the relevant features for the model
    """
    elastic = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, .99, 1],
                           alphas=None,
                           cv=None,
                           random_state=None)
    elastic.fit(data[x], data[y])
    coef = pd.Series(elastic.coef_, index=x)
    # return the selected features
    return coef[coef != 0].index.tolist()

def fit_regression_model(x: list[str], y: str, data: pd.DataFrame):
    """
    Fit a regression model to the data
    """
    log_regression = LogisticRegressionCV(
                            penalty='elasticnet',
                            solver='saga',
                            l1_ratio=[.1, .3, .5, .7, .9, .95, .99, 1])
    log_regression.fit(data[x], data[y])
    return log_regression


def get_historical_data(data: pd.DataFrame, x: list[str], y: str, time_window: int, current_date: date) -> pd.DataFrame:
    """
    Get the historical data for the given time window
    """
    start_date = current_date - relativedelta(months=time_window)
    full_columns = x + [y]
    return data[(data['Month'] >= start_date) & (data['Month'] < current_date)][full_columns]

def get_prediction(model: LogisticRegressionCV, row_vals: pd.Series) -> tuple:
    """
    Get the prediction for the given row
    """
    return (
        model.predict(row_vals),
        model.predict_proba(row_vals)
    )




